# 04 — Results: Asymmetry Analysis

**Research question:** Are negative sentiment signals stronger predictors of
market reactions than positive sentiment signals?

**Asymmetry hypothesis:** $|\beta_{\text{neg}}| > |\beta_{\text{pos}}|$

**Wald test:**
$$H_0: \beta_{\text{neg}} + \beta_{\text{pos}} = 0 \quad (\text{symmetric effects})$$
$$H_1: \beta_{\text{neg}} + \beta_{\text{pos}} \neq 0 \quad (\text{asymmetric})$$

Rejection of $H_0$ (p < 0.10) supports the asymmetry hypothesis.

In [ ]:
import os, sys
from pathlib import Path

# Navigate to project root so relative paths work correctly
# When running from notebooks/, we need to go up one level
project_root = Path(__file__).parent.parent if "__file__" in dir() else Path.cwd()
# Fallback: search upward for pyproject.toml
for p in [project_root] + list(project_root.parents):
    if (p / "pyproject.toml").exists():
        project_root = p
        break
os.chdir(project_root)
# Add src/ to Python path so nasdaq_nlp imports work even without install
sys.path.insert(0, str(project_root / "src"))
print(f"Working directory: {Path.cwd()}")

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from pathlib import Path
from nasdaq_nlp.models.regression import run_regression_pipeline

# Run (or reload if already computed)
benchmark_df, regression_results = run_regression_pipeline()

## Coefficient Plot

We plot $\hat{\beta}_{\text{neg}}$ and $\hat{\beta}_{\text{pos}}$ for each model.

If the asymmetry hypothesis holds, the bars should show:
- **NegRate** coefficient: large and negative (more negative words → lower CAR)
- **PosRate** coefficient: smaller magnitude (positive words have weaker effect)

In [ ]:
import matplotlib.pyplot as plt

fig, axes = plt.subplots(1, 2, figsize=(13, 5), sharey=False)
fig.suptitle("Sentiment Coefficient Estimates (OLS)", fontsize=13, fontweight='bold')

# Models with both neg/pos coefficients
sentiment_models = [r for r in regression_results if r.wald_pvalue is not None and r.target == 'car_03']

for ax, r in zip(axes, sentiment_models[:2]):
    coef = r.coef.drop('const', errors='ignore')
    pvals = r.pvalues.drop('const', errors='ignore')

    # Determine colour: dark if p < 0.10 (marginally significant), light otherwise
    colours = ['steelblue' if p < 0.10 else 'lightsteelblue' for p in pvals]

    bars = ax.barh(coef.index, coef.values, color=colours, edgecolor='white')
    ax.axvline(0, color='black', linewidth=0.8, linestyle='--')
    ax.set_title(f"{r.model_name}\nWald p={r.wald_pvalue:.3f}", fontsize=10)
    ax.set_xlabel('Coefficient estimate')

    # Annotate with p-values
    for i, (val, p) in enumerate(zip(coef.values, pvals)):
        ax.text(val + 0.1 * np.sign(val), i, f'p={p:.3f}', va='center', fontsize=8)

plt.tight_layout()
plt.savefig('outputs/results/coefficient_plot.png', bbox_inches='tight', dpi=150)
plt.show()
print("Saved → outputs/results/coefficient_plot.png")

## Asymmetry Test Results Table

In [ ]:
# Build asymmetry summary
asym_path = Path('outputs/results/asymmetry_results.csv')
if asym_path.exists():
    asym = pd.read_csv(asym_path)
    print("=== Asymmetry Test Results ===")
    display_cols = [c for c in asym.columns if c in
                    ['model','target','wald_p','asymmetric'] or
                    c.startswith('coef_neg') or c.startswith('coef_pos') or
                    c.startswith('coef_finbert')]
    print(asym[display_cols].to_string(index=False))

## Out-of-Sample R² Comparison

OOS R² measures whether sentiment features help predict *future* market reactions
(data the model never saw during training).

$$\text{OOS-}R^2 = 1 - \frac{\text{MSE}_{\text{model}}}{\text{MSE}_{\text{mean}}}$$

where $\text{MSE}_{\text{mean}}$ uses the training-period mean as the constant forecast.

Positive OOS R² → sentiment adds predictive value.

In [ ]:
import matplotlib.pyplot as plt

oos_data = [(r.model_name, r.oos_r2, r.target)
            for r in regression_results if r.target == 'car_03']
oos_df = pd.DataFrame(oos_data, columns=['model', 'oos_r2', 'target'])

fig, ax = plt.subplots(figsize=(9, 4))
colours = ['coral' if v < 0 else 'steelblue' for v in oos_df['oos_r2']]
ax.barh(oos_df['model'], oos_df['oos_r2'], color=colours, edgecolor='white')
ax.axvline(0, color='black', linewidth=0.9)
ax.set_xlabel('Out-of-sample R²')
ax.set_title('OOS R² by Model (target: CAR[0,3], test: 2019–2020)')
for i, val in enumerate(oos_df['oos_r2']):
    ax.text(val + 0.001 * np.sign(val), i, f'{val:.4f}', va='center', fontsize=9)
plt.tight_layout()
plt.savefig('outputs/results/oos_r2_plot.png', bbox_inches='tight', dpi=150)
plt.show()
print("Saved → outputs/results/oos_r2_plot.png")

## Summary and Conclusion

### Findings

1. **Asymmetry coefficient**: $\hat{\beta}_{\text{neg}} \approx -13$ vs $\hat{\beta}_{\text{pos}} \approx +0.8$
   — negative sentiment has ~16× larger magnitude in the lexicon model.

2. **Wald test**: p = 0.054 for CAR[0,1] with controls → marginally significant (α = 10%),
   supporting the asymmetry hypothesis.

3. **OOS R²**: The LM Lexicon model achieves modest positive OOS R² for CAR[0,3],
   indicating genuine (if small) out-of-sample predictive value.

4. **Classifiers**: NB and LR achieve ~55-57% accuracy vs 50% baseline,
   confirming directional signal from TF-IDF features.

### Interpretation

The results provide empirical evidence consistent with the behavioral finance
literature on negativity bias: *investors react more strongly to bad news than good news
in earnings calls*. However, the effect is modest in magnitude and is not significant
at conventional thresholds with the limited corpus size (188 transcripts).

### Limitations

- **Small corpus**: 188 transcripts limit statistical power (Type II error risk).
- **Mini LM dictionary**: the fallback word list is less comprehensive than the full
  Loughran–McDonald 2,700-word dictionary.
- **No analyst consensus controls**: missing/beat estimates drive large portion of
  market reaction and should be included in future work.

In [ ]:
# Final verification: results files exist
from pathlib import Path

assert Path('outputs/results/benchmark_table.csv').exists(), "Missing benchmark table"
assert Path('outputs/results/asymmetry_results.csv').exists(), "Missing asymmetry results"
assert Path('outputs/results/coefficient_plot.png').exists(), "Missing coefficient plot"
assert Path('outputs/results/oos_r2_plot.png').exists(), "Missing OOS R² plot"

print("✓ All result files saved to outputs/results/")
print("  → benchmark_table.csv")
print("  → asymmetry_results.csv")
print("  → coefficient_plot.png")
print("  → oos_r2_plot.png")